In [ ]:
# Import Rquired Libraries

import pandas as pd
from sqlalchemy import create_engine
import os
import logging
import time

# Create Logging Module

logging.basicConfig(
    filename ="logs/ingestion_db.log",
    level = logging.DEBUG,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a"

)

# Create Connection With MySQL

user = "root"
password = "sanket1234"
host = "localhost"
port = "3306"
database =  "delivery_performance"


try:
    engine = create_engine(
        f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
    )
except:
    print("Connection Error")

def ingestion_db(df, tablename, engine):
     df.to_sql(tablename, con = engine , if_exists  = 'replace', index = False)

def load_row_data():

    start = time.time()
    for file in os.listdir('data_files'):
        if '.csv' in file:
            print(file)


            df = pd.read_csv('data_files/'+file)
            logging.info(f"ingesting {file} in database")
            ingestion_db(df, file[:-4], engine)
    end = time.time()

    total_time = (end - start) / 60
    logging.info("ingestion complete")
    logging.info(f"\n Total Time Taken: {total_time} minutes")

if __name__  == '__main__':
    load_row_data()





In [ ]:
tables = pd.read_sql_query("show tables from delivery_performance", con = engine)
tables

In [ ]:
for table in tables['Tables_in_delivery_performance']:
    print('-' * 70 , f'{table}', '-' * 70)
    print('Count of Records : ', pd.read_sql_query(f"SELECT COUNT(*) AS Count FROM {table}", con = engine)['Count'].values[0])
    display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", con = engine))


In [ ]:
pd.read_sql_query("""
    select 
        count(order_id) as order_id, 
        count(distinct(order_id)) as unique_id, 
        (count(order_id) - count(distinct(order_id))) as repeated_customers 
    from order_items;
""", con = engine)

In [ ]:
pd.read_sql_query("""
    select 
        count(order_id) as order_id, 
        count(distinct(order_id)) as unique_id, 
        (count(order_id) - count(distinct(order_id))) as repeated_customers 
    from orders;""", con =engine
)

In [ ]:
pd.read_sql_query("""
    select 
        count(order_id) as order_id, 
        count(distinct(order_id)) as unique_id, 
        (count(order_id) - count(distinct(order_id))) as repeated_customers 
    from payments;""", con =engine
)

the number of customers comes to order_items are 112650 of which unique are 98666 and repeated customers are 13984.
the number of customer click order are 99441 of which unique are 99441 and repeated customer none .
the number of customer done payments are 103886 of which unique are 99440 and customer repeated are 4446.

In [ ]:
pd.read_sql_query("""
    select 
        count(distinct(ge.geolocation_state)) as geolocation_state, 
        count(distinct( seller_state)) as seller_states, 
        (count(distinct(ge.geolocation_state))-count(distinct(se.seller_state))) as no_seller,
        count(distinct(ge.geolocation_city)) as geolocation_city, 
        count(distinct( seller_city)) as seller_city, 
        (count(distinct(ge.geolocation_city))-count(distinct(se.seller_city))) as no_seller_city,
        count(distinct(ge.geolocation_zip_code_prefix)) as geolocation_zip_code_prefix, 
        count(distinct( seller_zip_code_prefix)) as seller_zip_code_prefix, 
        (count(distinct(ge.geolocation_zip_code_prefix))-count(distinct(se.seller_zip_code_prefix))) as no_seller_zip
    from geolocation as ge left join sellers as se 
    on  ge.geolocation_zip_code_prefix = se.seller_zip_code_prefix; """, con = engine
)

In [ ]:
pd.read_sql_query("""
SELECT geolocation_state FROM geolocation
EXCEPT
SELECT seller_state FROM sellers;
""", con = engine)

In [ ]:
pd.read_sql_query("""select order_status, count(order_status) as total_orders from orders group by order_status""", con = engine)

In [ ]:
pd.read_sql_query(""" select * from products""", con = engine)

Combine All Tables TO Make a Single Table For Better Analysis ,
Joined All The Tables On Inner Join,  
TO Make Sure All Data Available In Both Table Will Available Into Single Table

In [ ]:
# This Table Created by Combination of 5 different Tables of Which The Center or The Main Table Is Orders

df = pd.read_sql_query("""
select 
    ords.order_id,
    ords.order_status,
    ords.order_purchase_timestamp,
    ords.order_approved_at,
    ords.order_delivered_customer_date,
    ords.order_delivered_carrier_date,
    ords.order_estimated_delivery_date,
    ord.seller_id,
    ord.shipping_limit_date,
    py.payment_type,
    py.payment_installments,
    sum(py.payment_value) as total_payment_value,
    pd.product_category,
    pd.product_id,
    pd.product_weight_g,
    se.seller_city,
    se.seller_state,
    sum(ord.price) as total_price,
    sum(ord.freight_value) as total_freight_value
                        
from orders as ords inner join order_items as ord
on ords.order_id = ord.order_id
inner join payments as py
on ords.order_id = py.order_id
inner join products as pd
on ord.product_id = pd.product_id
inner join sellers as se
on ord.seller_id = se.seller_id

group by 
    order_id,
    ords.order_status,
    ords.order_purchase_timestamp,
    ords.order_approved_at,
    ords.order_delivered_customer_date,
    order_delivered_carrier_date,
    ords.order_estimated_delivery_date,
    ord.seller_id,
    ord.shipping_limit_date,
    py.payment_type,
    py.payment_installments,
    pd.product_category,
    pd.product_id,
    pd.product_weight_g,
    se.seller_city,
    se.seller_state
""", con = engine)

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

There Are Null Values in product category, product weight , order_delivered_at , order_delivered_carrier_date, order_approved_at and order_delivered_to_customer_date as This Data is Inconsistance , Need TO Remove This Data to Have a clean Data For Better Analysis

In [ ]:
df.dropna(subset = ['product_category', 'product_weight_g', 'order_approved_at','order_delivered_customer_date', 'order_delivered_carrier_date' ], inplace = True)

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
df = df.drop(columns = ['product_id'])

In [ ]:
df.head(10)

In [ ]:
new_order = ['order_id', 'seller_id', 'product_category','product_weight_g', 'order_status', 'order_purchase_timestamp','order_approved_at', 'order_delivered_customer_date','order_delivered_carrier_date', 'order_estimated_delivery_date', 'shipping_limit_date','payment_installments', 'payment_type', 'total_freight_value', 'total_price','total_payment_value','seller_city', 'seller_state'   ]

In [ ]:
df = df[new_order]

all null values are replaced now converting the data to correct data types


In [ ]:
df.dtypes

In [ ]:
date_column = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'shipping_limit_date', 'order_delivered_carrier_date']

for col in date_column:
    df[col] = pd.to_datetime(df[col]).dt.date

df[date_column] = df[date_column].apply(pd.to_datetime, errors = 'coerce')

In [ ]:
df['order_handling_days'] = (df['order_delivered_carrier_date'] - df['order_approved_at']).dt.days

In [ ]:
df['order_delivered_customer_days'] = (df['order_delivered_customer_date'] - df['order_approved_at']).dt.days

In [ ]:
df['order_approved_days'] = (df['order_approved_at'] - df['order_purchase_timestamp']).dt.days

In [ ]:
df['order_delivered_late'] =( df['order_delivered_customer_date'] > df['order_estimated_delivery_date'])

In [ ]:
df['is_late_shipping'] = (df['order_delivered_carrier_date'] > df['shipping_limit_date'])

In [ ]:
df.dtypes

In [ ]:
product_delivery_summary = df

In [ ]:
table_name = 'product_delivery_summary'

df.to_sql(name = table_name, con = engine, if_exists = 'replace', index = False, chunksize = 1000)

In [ ]:
pd.read_sql_query("""select * from product_delivery_summary """, con = engine)